# Прогон B — `refactor/logging` + DagsHub

In [1]:
import os, sys, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

RUN_NAME = "v3-logging"

sec = UserSecretsClient()
for name in ("GIT_URL", "GITHUB_TOKEN", "GIT_BRANCH", "DAGSHUB_TOKEN"):
    try:
        os.environ[name] = sec.get_secret(name)
    except Exception:
        pass

REPO = Path("/kaggle/working/filament-segmentation-2026")
BRANCH = os.environ.get("GIT_BRANCH", "refactor/logging")
url = os.environ.get("GIT_URL", "")
token = os.environ.get("GITHUB_TOKEN", "")
if token and url.startswith("https://"):
    url = url.replace("https://", f"https://{token}@", 1)

if not (REPO / "run.py").exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)])
else:
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "checkout", "-B", BRANCH, f"origin/{BRANCH}"])

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True))

!pip -q install transformers accelerate safetensors python-dotenv mlflow dagshub matplotlib

dagshub_token = os.environ.get("DAGSHUB_TOKEN", "").strip()
if not dagshub_token:
    raise SystemExit(
        "Add Kaggle secret DAGSHUB_TOKEN: DagsHub → Settings → Tokens → Create"
    )

env_lines = Path(".env.example").read_text().splitlines()
out_lines = []
for line in env_lines:
    if line.startswith("DAGSHUB_TOKEN="):
        out_lines.append(f"DAGSHUB_TOKEN={dagshub_token}")
    else:
        out_lines.append(line)
Path(".env").write_text("\n".join(out_lines) + "\n")
print("DagsHub: on → https://dagshub.com/BraveCactus/solar-filament-segmentation")
print("MLflow run name:", RUN_NAME)

Cloning into '/kaggle/working/filament-segmentation-2026'...


1656875 Change confidence threshhold & add kaggle notebook for uv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 85.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.0/274.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 

In [2]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Settings → GPU T4, Restart session"
x = torch.zeros(1, device="cuda")
print("ok", x.device)

True Tesla T4
ok cuda:0


In [3]:
%cd /kaggle/working/filament-segmentation-2026
!python run.py check
!python run.py train --size 1024 --epochs 8 --run-name {RUN_NAME} --eval-samples-val 100 --eval-samples-train 200 --augment
!ls -lh /kaggle/working/outputs/m2f/last.pt /kaggle/working/outputs/m2f/experiments/{RUN_NAME}/best.pt

/kaggle/working
check ok
preprocessor_config.json: 100%|████████████████| 537/537 [00:00<00:00, 3.60MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
config.json: 79.8kB [00:00, 61.3MB/s]
model.safetensors: 100%|█████████████████████| 190M/190M [00:02<00:00, 93.2MB/s]
Loading weights: 100%|█| 566/566 [00:00<00:00, 1487.90it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
criterion.empty_weight | MISMATCH | Reinit due to size mismatch ckpt: torch

In [4]:
from pathlib import Path
RUN = Path(f"/kaggle/working/outputs/m2f/experiments/{RUN_NAME}")
print("metrics.csv:")
print((RUN / "metrics.csv").read_text())
print("\nweights:", sorted((RUN / "weights").glob("*.pt")))
print("good:", len(list((RUN / "good").glob("*.png"))), "bad:", len(list((RUN / "bad").glob("*.png"))))

metrics.csv:
epoch,train_loss,train_pq,val_pq
1,22.09937218643696,0.2529820676154284,0.2760671031796554
2,19.841290428865374,0.24955950593352985,0.25285113767710227
3,18.906059519408423,0.27984411150471444,0.29090282390815136
4,18.717923045624655,0.27910775801488336,0.2827983512386906
5,19.180583618688146,0.277967059013521,0.27795623800250346
6,18.49364654290474,0.2747600454571971,0.2884133113220034
7,18.158794625175357,0.29500284747377625,0.30528503451331507
8,18.33260411606929,0.27685579483350614,0.28489649378003506


weights: [PosixPath('/kaggle/working/outputs/m2f/experiments/v3-logging/weights/epoch_1.pt'), PosixPath('/kaggle/working/outputs/m2f/experiments/v3-logging/weights/epoch_2.pt'), PosixPath('/kaggle/working/outputs/m2f/experiments/v3-logging/weights/epoch_3.pt'), PosixPath('/kaggle/working/outputs/m2f/experiments/v3-logging/weights/epoch_4.pt'), PosixPath('/kaggle/working/outputs/m2f/experiments/v3-logging/weights/epoch_5.pt'), PosixPath('/kaggle/working/outputs/m2f/exper

In [5]:
# Save Version с Output (~200MB) перед infer, если сессия оборвётся
%cd /kaggle/working/filament-segmentation-2026
!python run.py infer --submit --weights /kaggle/working/outputs/m2f/experiments/{RUN_NAME}/best.pt
!ls -lh /kaggle/working/outputs/m2f/submission_m2f.csv

/kaggle/working
/usr/local/lib/python3.12/dist-packages/transformers/image_processing_base.py:370: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)
Loading weights: 100%|█| 566/566 [00:00<00:00, 1591.37it/s, Materializing param=
Mask2FormerForUniversalSegmentation LOAD REPORT from: facebook/mask2former-swin-tiny-coco-instance
Key                    | Status   |                                                                                      
-----------------------+----------+--------------------------------------------------------------------------------------
criterion.empty_weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81]) vs model:torch.Size([2])          
class_predictor.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([81, 256]) vs model:torch.Size([2, 256])
class_predictor.bias   | MISMATCH | Rein